# Module 9: Causal Discovery and Markov Blanket Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Run the PC algorithm on a real dataset and visualise the discovered causal graph
2. Extract the Markov blanket of a target variable from the CPDAG
3. Compare Markov blanket features against Lasso, Boruta, and SHAP selections
4. Understand when causal and predictive feature sets agree and when they diverge

## Prerequisites

- Module 9 Guide 01: Causal Graphs and the Markov Blanket
- Understanding of conditional independence and partial correlation
- Basic familiarity with DAGs and graphical models

## Estimated Time: 15 minutes

## Libraries Required
```
pip install causal-learn shap boruta-py scikit-learn pandas matplotlib networkx
```

## 1. Setup

In [ ]:
# Purpose: Import all libraries and configure the environment
# Key concept: causal-learn provides PC, FCI, and GES implementations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

# Causal discovery
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz, kci
from causallearn.search.ScoreBased.GES import ges

# Standard feature selection
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.datasets import fetch_california_housing
import shap

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("Setup complete.")

## 2. Dataset: California Housing

We use the California Housing dataset (20,640 samples, 8 features) to discover the causal structure among features and the target variable `MedHouseVal`.

The 8 features are:
- **MedInc**: Median income in block group
- **HouseAge**: Median house age in block group
- **AveRooms**: Average number of rooms per household
- **AveBedrms**: Average number of bedrooms per household
- **Population**: Block group population
- **AveOccup**: Average number of household members
- **Latitude**: Block group latitude
- **Longitude**: Block group longitude

We expect a causal chain: location (Lat/Long) → demographics (Income, Population) → housing characteristics (Rooms, Bedrms) → price.

In [ ]:
# Purpose: Load and prepare the California housing dataset
# Key concept: real data with known partial causal structure for validation

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
feature_names = list(housing.feature_names) + ['MedHouseVal']

# Use a representative subsample for PC (full dataset is slow for CI tests)
df_sample = df.sample(n=2000, random_state=42).reset_index(drop=True)

# Standardise for causal discovery (PC assumes zero-mean data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sample.values)
df_scaled = pd.DataFrame(X_scaled, columns=feature_names)

print(f"Dataset: {df_sample.shape[0]} samples, {len(feature_names)} variables")
print(f"Features: {feature_names}")
print(f"\nCorrelation with target (MedHouseVal):")
for col in housing.feature_names:
    corr = df_sample[col].corr(df_sample['MedHouseVal'])
    print(f"  {col:12s}: {corr:+.3f}")

## 3. Running the PC Algorithm

The PC algorithm discovers the skeleton (which variables are connected) and then orients edges using v-structure rules.

**Key parameters:**
- `alpha`: Significance level for conditional independence tests (lower = more conservative = fewer edges removed = denser graph)
- `indep_test`: The CI test to use (`fisherz` for Gaussian continuous data)
- `stable`: Use stable PC (consistent regardless of variable ordering)

In [ ]:
# Purpose: Run PC algorithm to discover causal structure
# Key concept: PC iteratively removes edges using conditional independence tests

print("Running PC algorithm (this takes ~30 seconds)...")

# Run PC with Fisher-z test (appropriate for approximately Gaussian data)
cg = pc(
    data=X_scaled,
    alpha=0.01,           # Strict significance level — fewer false edges
    indep_test=fisherz,   # Partial correlation test for Gaussian data
    stable=True,          # Stable PC: consistent across variable orderings
    uc_rule=0,            # Majority rule for v-structure orientation
    uc_priority=-1        # Lower priority for existing orientations
)

print("PC algorithm complete.")
print(f"\nEdges in discovered graph:")

# The graph adjacency matrix:
# cg.G.graph[i,j] == -1 and cg.G.graph[j,i] == 1 means i -> j
# cg.G.graph[i,j] == 1 and cg.G.graph[j,i] == 1 means i -- j (undirected)
# cg.G.graph[i,j] == 0 means no edge i - j
adj = cg.G.graph
p = len(feature_names)

edge_list = []
for i in range(p):
    for j in range(i+1, p):
        if adj[i,j] != 0 or adj[j,i] != 0:
            if adj[i,j] == -1 and adj[j,i] == 1:
                edge_str = f"  {feature_names[i]} → {feature_names[j]}"
                edge_list.append(('directed', i, j))
            elif adj[j,i] == -1 and adj[i,j] == 1:
                edge_str = f"  {feature_names[j]} → {feature_names[i]}"
                edge_list.append(('directed', j, i))
            else:
                edge_str = f"  {feature_names[i]} — {feature_names[j]} (undirected)"
                edge_list.append(('undirected', i, j))
            print(edge_str)

## 4. Visualising the Causal Graph

We convert the discovered CPDAG to a NetworkX graph for visualisation. Directed edges are drawn with arrows; undirected edges (ambiguous orientation) are drawn as dashed lines.

In [ ]:
# Purpose: Visualise the discovered causal graph
# Key concept: NetworkX makes it easy to render the CPDAG structure

def plot_causal_graph(adj: np.ndarray, feature_names: list, target_name: str):
    """
    Visualise the discovered CPDAG from PC algorithm.

    Parameters
    ----------
    adj : np.ndarray
        Adjacency matrix from causal-learn (cg.G.graph).
    feature_names : list
        Names of all variables including target.
    target_name : str
        Name of the target variable (highlighted in blue).
    """
    G_directed = nx.DiGraph()
    G_undirected = nx.Graph()
    G_directed.add_nodes_from(feature_names)
    G_undirected.add_nodes_from(feature_names)

    p = len(feature_names)
    for i in range(p):
        for j in range(i+1, p):
            if adj[i,j] == -1 and adj[j,i] == 1:
                G_directed.add_edge(feature_names[i], feature_names[j])
            elif adj[j,i] == -1 and adj[i,j] == 1:
                G_directed.add_edge(feature_names[j], feature_names[i])
            elif adj[i,j] == 1 and adj[j,i] == 1:
                G_undirected.add_edge(feature_names[i], feature_names[j])

    fig, ax = plt.subplots(figsize=(12, 8))

    # Use spring layout for good node placement
    pos = nx.spring_layout(G_directed, seed=42, k=2.5)

    # Node colours: target in blue, features in light green
    node_colors = ['#4a90d9' if n == target_name else '#90c96b'
                   for n in G_directed.nodes()]

    # Draw directed edges
    nx.draw_networkx_edges(G_directed, pos, ax=ax,
                           arrowstyle='->', arrowsize=20,
                           edge_color='#333333', width=2,
                           connectionstyle='arc3,rad=0.05')

    # Draw undirected edges (dashed)
    nx.draw_networkx_edges(G_undirected, pos, ax=ax,
                           edge_color='#888888', width=1.5,
                           style='dashed')

    # Draw nodes
    nx.draw_networkx_nodes(G_directed, pos, ax=ax,
                           node_color=node_colors, node_size=2500,
                           alpha=0.95)
    nx.draw_networkx_labels(G_directed, pos, ax=ax, font_size=9, font_weight='bold')

    # Legend
    legend_handles = [
        mpatches.Patch(facecolor='#4a90d9', label='Target variable'),
        mpatches.Patch(facecolor='#90c96b', label='Feature'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=10)
    ax.set_title('Discovered Causal Graph (PC Algorithm, alpha=0.01)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_causal_graph(adj, feature_names, target_name='MedHouseVal')

## 5. Extracting the Markov Blanket

The Markov blanket of `MedHouseVal` consists of:
- **Parents:** variables with directed edges pointing *to* MedHouseVal
- **Children:** variables with directed edges pointing *from* MedHouseVal
- **Co-parents:** other parents of MedHouseVal's children

For undirected edges, we include the adjacent node conservatively.

In [ ]:
# Purpose: Extract the Markov blanket from the discovered CPDAG
# Key concept: MB(Y) = parents ∪ children ∪ co-parents

def extract_markov_blanket(adj: np.ndarray, target_idx: int,
                            feature_names: list) -> dict:
    """
    Extract Markov blanket from PC algorithm CPDAG.

    causal-learn adjacency encoding:
      adj[i,j] == -1 and adj[j,i] == 1: i -> j
      adj[i,j] == 1 and adj[j,i] == 1: i -- j (undirected)
      adj[i,j] == 0: no edge

    Parameters
    ----------
    adj : np.ndarray
        Adjacency matrix from cg.G.graph.
    target_idx : int
        Index of the target variable.
    feature_names : list
        Names of all variables.

    Returns
    -------
    dict with keys 'parents', 'children', 'co_parents', 'mb'
    """
    p = len(feature_names)
    parents, children, undirected_neighbors = [], [], []

    t = target_idx
    for i in range(p):
        if i == t:
            continue
        # i -> t: adj[i,t] == -1 and adj[t,i] == 1
        if adj[i, t] == -1 and adj[t, i] == 1:
            parents.append(i)
        # t -> i: adj[t,i] == -1 and adj[i,t] == 1
        elif adj[t, i] == -1 and adj[i, t] == 1:
            children.append(i)
        # undirected: adj[i,t] == 1 and adj[t,i] == 1
        elif adj[i, t] == 1 and adj[t, i] == 1:
            undirected_neighbors.append(i)  # Conservative: include as possible parent
            parents.append(i)

    # Co-parents: other parents of each child of target
    co_parents = []
    for c in children:
        for j in range(p):
            if j == t or j == c:
                continue
            # j -> c
            if adj[j, c] == -1 and adj[c, j] == 1:
                co_parents.append(j)

    mb_indices = list(set(parents + children + co_parents))
    mb_names = [feature_names[i] for i in mb_indices]

    return {
        'parents': [feature_names[i] for i in parents],
        'children': [feature_names[i] for i in children],
        'co_parents': [feature_names[i] for i in co_parents],
        'undirected': [feature_names[i] for i in undirected_neighbors],
        'mb': mb_names,
    }

# Target: MedHouseVal (last column, index 8)
target_idx = feature_names.index('MedHouseVal')
mb_result = extract_markov_blanket(adj, target_idx, feature_names)

print("=" * 50)
print("Markov Blanket of MedHouseVal")
print("=" * 50)
print(f"\nParents (direct causes): {mb_result['parents']}")
print(f"Children (direct effects): {mb_result['children']}")
print(f"Co-parents (other causes of children): {mb_result['co_parents']}")
print(f"Undirected neighbors (ambiguous): {mb_result['undirected']}")
print(f"\nMarkov Blanket MB(MedHouseVal): {mb_result['mb']}")
print(f"\nFeatures NOT in MB: {[f for f in housing.feature_names if f not in mb_result['mb']]}")

## 6. Standard Feature Selection for Comparison

Now we run Lasso, Boruta, and SHAP-based feature selection on the same dataset and compare with the Markov blanket.

In [ ]:
# Purpose: Run standard feature selection methods for comparison
# Key concept: compare predictive selection with causal (MB) selection

X = df_sample[housing.feature_names].values
y = df_sample['MedHouseVal'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# ── Lasso selection ─────────────────────────────────────────────────────────
lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
lasso.fit(X_train_sc, y_train)
lasso_features = [housing.feature_names[i]
                  for i in np.where(np.abs(lasso.coef_) > 1e-4)[0]]
print(f"Lasso selected ({len(lasso_features)}): {lasso_features}")

# ── SHAP selection (from Gradient Boosting) ──────────────────────────────────
gb_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42
)
gb_model.fit(X_train_sc, y_train)

explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_train_sc)
shap_importance = np.abs(shap_values).mean(axis=0)

# Select features with above-mean SHAP importance
mean_shap = shap_importance.mean()
shap_features = [housing.feature_names[i]
                 for i in np.where(shap_importance > mean_shap)[0]]
print(f"SHAP selected ({len(shap_features)}): {shap_features}")

# ── Random Forest importance ──────────────────────────────────────────────────
rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(X_train_sc, y_train)
rf_importance = rf_model.feature_importances_
mean_rf_imp = rf_importance.mean()
rf_features = [housing.feature_names[i]
               for i in np.where(rf_importance > mean_rf_imp)[0]]
print(f"RF importance ({len(rf_features)}): {rf_features}")

print(f"\nMarkov Blanket (PC): {mb_result['mb']}")

## 7. Comparing Feature Sets

We visualise the overlap between the Markov blanket and standard selection methods, then evaluate the predictive performance of each feature set.

In [ ]:
# Purpose: Compare and visualise overlap between feature sets
# Key concept: Jaccard similarity measures how much methods agree

feature_sets = {
    'Markov Blanket\n(PC)': set(mb_result['mb']),
    'Lasso': set(lasso_features),
    'SHAP\n(GradBoost)': set(shap_features),
    'RF\nImportance': set(rf_features),
}

all_features = set(housing.feature_names)

# Membership matrix: each row is a feature, each column is a method
method_names = list(feature_sets.keys())
membership = np.zeros((len(housing.feature_names), len(method_names)))
for j, (method, fset) in enumerate(feature_sets.items()):
    for i, feat in enumerate(housing.feature_names):
        membership[i, j] = feat in fset

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Feature membership heatmap
import matplotlib.colors as mcolors
cmap = mcolors.ListedColormap(['#f0f0f0', '#4a90d9'])
im = axes[0].imshow(membership, aspect='auto', cmap=cmap, vmin=0, vmax=1)
axes[0].set_xticks(range(len(method_names)))
axes[0].set_xticklabels(method_names, fontsize=10)
axes[0].set_yticks(range(len(housing.feature_names)))
axes[0].set_yticklabels(housing.feature_names, fontsize=10)
axes[0].set_title('Feature Selection: Method Comparison', fontsize=12)
for i in range(len(housing.feature_names)):
    for j in range(len(method_names)):
        axes[0].text(j, i, 'Yes' if membership[i,j] else 'No',
                    ha='center', va='center', fontsize=8,
                    color='white' if membership[i,j] else '#888888')

# Plot 2: Jaccard similarity between methods
n_methods = len(method_names)
jaccard_matrix = np.zeros((n_methods, n_methods))
method_sets = list(feature_sets.values())
for i in range(n_methods):
    for j in range(n_methods):
        inter = len(method_sets[i] & method_sets[j])
        union = len(method_sets[i] | method_sets[j])
        jaccard_matrix[i, j] = inter / union if union > 0 else 1.0

im2 = axes[1].imshow(jaccard_matrix, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks(range(n_methods))
axes[1].set_xticklabels(method_names, fontsize=10)
axes[1].set_yticks(range(n_methods))
axes[1].set_yticklabels(method_names, fontsize=10)
axes[1].set_title('Jaccard Similarity Between Methods', fontsize=12)
plt.colorbar(im2, ax=axes[1], shrink=0.8)
for i in range(n_methods):
    for j in range(n_methods):
        axes[1].text(j, i, f'{jaccard_matrix[i,j]:.2f}',
                    ha='center', va='center', fontsize=10,
                    color='white' if jaccard_matrix[i,j] < 0.4 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# Purpose: Evaluate predictive performance of each feature set
# Key concept: causal features may sacrifice some in-distribution accuracy

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

def evaluate_feature_set(feature_names_subset, X_train, X_test, y_train, y_test,
                         all_feature_names):
    """Evaluate R2 for a feature subset using gradient boosting."""
    indices = [list(all_feature_names).index(f) for f in feature_names_subset]
    if not indices:
        return np.nan
    X_tr = X_train[:, indices]
    X_te = X_test[:, indices]
    model = GradientBoostingRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42
    )
    model.fit(X_tr, y_train)
    return r2_score(y_test, model.predict(X_te))

results = {}
all_feat_names = list(housing.feature_names)

for method_name, fset in feature_sets.items():
    feat_list = list(fset)
    if not feat_list:
        results[method_name] = np.nan
        continue
    r2 = evaluate_feature_set(feat_list, X_train_sc, X_test_sc,
                               y_train, y_test, all_feat_names)
    results[method_name] = r2
    clean_name = method_name.replace('\n', ' ')
    print(f"{clean_name:20s}: R2={r2:.4f}  ({len(fset)} features)")

# Baseline: all features
all_r2 = evaluate_feature_set(all_feat_names, X_train_sc, X_test_sc,
                               y_train, y_test, all_feat_names)
print(f"{'All features':20s}: R2={all_r2:.4f}  ({len(all_feat_names)} features)")

## 8. Visualising the Performance-Sparsity Trade-off

In [ ]:
# Purpose: Plot performance vs feature count for each method
# Key concept: causal methods give different sparsity-accuracy trade-offs

fig, ax = plt.subplots(figsize=(9, 6))

colors = {
    'Markov Blanket\n(PC)': '#4a90d9',
    'Lasso': '#e74c3c',
    'SHAP\n(GradBoost)': '#27ae60',
    'RF\nImportance': '#f39c12',
}

for method_name, fset in feature_sets.items():
    r2 = results[method_name]
    n_feat = len(fset)
    color = colors[method_name]
    clean_name = method_name.replace('\n', ' ')
    ax.scatter(n_feat, r2, s=200, color=color, zorder=5, label=clean_name)
    ax.annotate(clean_name, (n_feat, r2),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

# Add all-features baseline
ax.axhline(y=all_r2, color='grey', linestyle='--', alpha=0.7, label='All features baseline')
ax.text(0.5, all_r2 + 0.005, f'All features (R²={all_r2:.3f})', fontsize=9, color='grey')

ax.set_xlabel('Number of Selected Features', fontsize=12)
ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Feature Selection: Accuracy vs Sparsity', fontsize=13)
ax.set_xlim(0, len(housing.feature_names) + 1)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Features near the top-left are ideal: high accuracy with few features")
print("- Markov blanket from PC captures causal structure; may sacrifice some accuracy")
print("- Predictive methods (Lasso, SHAP) optimise accuracy but include spurious correlates")
print("- Under distribution shift, causal features maintain performance while predictive degrade")

## 9. Exercise: GES Algorithm and Feature Comparison

**Task:** Run GES (Greedy Equivalence Search) on the same dataset and compare its Markov blanket with the PC result.

**GES uses a score-based approach (BIC) rather than CI tests.** It should be faster and potentially more sample-efficient.

**Questions to answer:**
1. Does GES find the same Markov blanket as PC?
2. Which edges differ between the PC and GES CPDAGs?
3. Which feature set gives better test R²?

**Hints:**
- Use `from causallearn.search.ScoreBased.GES import ges`
- GES syntax: `record = ges(X_scaled, score_func='local_score_BIC')`
- The graph is in `record['G'].graph` (same format as PC)

In [ ]:
# YOUR CODE HERE
# ──────────────────────────────────────────────────────────────────
# Step 1: Run GES algorithm

# Step 2: Extract adjacency matrix from GES output

# Step 3: Extract Markov blanket for MedHouseVal using extract_markov_blanket()

# Step 4: Compare GES MB with PC MB

# Step 5: Evaluate GES MB features using evaluate_feature_set()

In [ ]:
# AUTO-CHECK — validates your exercise solution
def test_exercise_ges():
    try:
        # Check GES was attempted
        assert 'record' in dir() or 'ges_record' in dir(), \
            "Run GES: record = ges(X_scaled, score_func='local_score_BIC')"
        print("Exercise check: GES has been run.")
        print("Verify: did GES find the same features as PC?")
        print("If yes: strong evidence for the Markov blanket.")
        print("If no: investigate which algorithm's assumptions fit the data better.")
    except AssertionError as e:
        print(f"Not yet complete: {e}")

test_exercise_ges()

## 10. Summary

### Key Takeaways

1. **PC algorithm** discovers causal structure via conditional independence tests. The result is a CPDAG — some edges may be undirected (ambiguous orientation).
2. **Markov blanket** = parents ∪ children ∪ co-parents. It is the theoretically optimal feature set under the true causal model.
3. **Causal and predictive selection can diverge.** The Markov blanket may omit features that correlate with $Y$ via confounders, and may include features (children, co-parents) that standard methods miss.
4. **In-distribution performance** of causal features is often similar to predictive methods — the difference emerges under distribution shift.
5. **GES** provides a score-based alternative to PC — generally more sample-efficient for Gaussian data.

### What's Next

Notebook 02 introduces Invariant Causal Prediction (ICP) — a method that identifies causal features without recovering the full graph, using only the invariance property across environments.

### References
- Pearl, J. (2009). *Causality* (2nd ed.). Cambridge University Press.
- Spirtes, P., Glymour, C., & Scheines, R. (2000). *Causation, Prediction, and Search* (2nd ed.). MIT Press.
- Chickering, D. M. (2002). Optimal structure identification with greedy search. *JMLR*, 3, 507–554.
- `causal-learn` documentation: https://causal-learn.readthedocs.io